# LLM-as-Judge Evaluation

Use GPT-4 to evaluate generated medical reports.

In [ ]:
from openai import OpenAI
import json
import pandas as pd
from typing import Dict, List

## Report Judge Class

In [ ]:
class ReportJudge:
    """GPT-4 based judge for medical report quality."""
    
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key)
    
    def evaluate(self, report, ground_truth_data):
        """
        Evaluate a generated report.
        
        Args:
            report: Generated report text
            ground_truth_data: Dict with abnormality_score, disease_probs, retrieved_cases
        
        Returns:
            dict: Evaluation scores and justifications
        """
        prompt = self._build_judge_prompt(report, ground_truth_data)
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are an expert radiologist evaluating AI-generated reports."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        
        return json.loads(response.choices[0].message.content)
    
    def _build_judge_prompt(self, report, ground_truth):
        """Build evaluation prompt."""
        
        # Format disease probabilities
        disease_str = "\n".join([
            f"  • {disease}: {prob:.3f}"
            for disease, prob in sorted(ground_truth['disease_probs'].items(), 
                                       key=lambda x: x[1], reverse=True)[:5]
            if prob > 0.1
        ])
        
        # Format retrieved cases
        retrieved_str = ""
        for i, case in enumerate(ground_truth.get('retrieved_cases', {}).get('abnormal', [])[:2], 1):
            retrieved_str += f"\nCase {i}: {case.get('impression', 'N/A')[:100]}"
        
        return f"""Evaluate the following chest X-ray report on a scale of 1-5 for each criterion:

GROUND TRUTH DATA:
- Abnormality Score: {ground_truth['abnormality_score']:.3f}
- Top Disease Probabilities:
{disease_str}
- Retrieved Similar Cases:{retrieved_str}

GENERATED REPORT:
{report}

EVALUATION CRITERIA (1=Poor, 5=Excellent):

1. Clinical Accuracy: Are findings consistent with disease probabilities?
2. Evidence Grounding: Are claims supported by retrieved cases?
3. Completeness: Does it cover all relevant findings?
4. Clarity: Is language clear and professional?
5. Appropriate Caution: Does it avoid overconfident claims?

Provide scores and brief justifications in JSON format:
{{
  "clinical_accuracy": {{"score": X, "justification": "..."}},
  "evidence_grounding": {{"score": X, "justification": "..."}},
  "completeness": {{"score": X, "justification": "..."}},
  "clarity": {{"score": X, "justification": "..."}},
  "caution": {{"score": X, "justification": "..."}},
  "overall_score": X.X,
  "summary": "..."
}}
"""

## Initialize Judge

In [ ]:
# Replace with your OpenAI API key
API_KEY = "your-openai-api-key"

judge = ReportJudge(api_key=API_KEY)

## Test Evaluation

In [ ]:
# Example report and ground truth
test_report = """FINDINGS:
The cardiac silhouette is enlarged, consistent with cardiomegaly. 
Small bilateral pleural effusions are present. 
No focal consolidation or pneumothorax.

COMPARISON:
Similar findings to retrieved abnormal case showing congestive heart failure pattern.

IMPRESSION:
Cardiomegaly with bilateral pleural effusions, suggestive of congestive heart failure.
"""

ground_truth = {
    'abnormality_score': 0.68,
    'disease_probs': {
        'Cardiomegaly': 0.72,
        'Pleural Effusion': 0.45,
        'Atelectasis': 0.31
    },
    'retrieved_cases': {
        'abnormal': [
            {'impression': 'Congestive heart failure with cardiomegaly and effusions.'}
        ]
    }
}

# Evaluate
scores = judge.evaluate(test_report, ground_truth)

# Display results
print("EVALUATION RESULTS:")
print("="*60)
for criterion, data in scores.items():
    if isinstance(data, dict):
        print(f"\n{criterion.upper()}:")
        print(f"  Score: {data['score']}/5")
        print(f"  Justification: {data['justification']}")
    else:
        print(f"\n{criterion.upper()}: {data}")

## Batch Evaluation

In [ ]:
def evaluate_batch(judge, reports_data, output_path='results/llm_judge_scores.json'):
    """
    Evaluate multiple reports.
    
    Args:
        judge: ReportJudge instance
        reports_data: List of dicts with 'report' and 'ground_truth'
        output_path: Where to save results
    
    Returns:
        list: Evaluation results
    """
    all_scores = []
    
    for i, data in enumerate(reports_data, 1):
        print(f"Evaluating report {i}/{len(reports_data)}...")
        
        try:
            scores = judge.evaluate(data['report'], data['ground_truth'])
            scores['case_id'] = data.get('case_id', i)
            all_scores.append(scores)
        
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    
    # Save results
    with open(output_path, 'w') as f:
        json.dump(all_scores, f, indent=2)
    
    print(f"\n✅ Evaluated {len(all_scores)} reports")
    print(f"✅ Results saved to {output_path}")
    
    return all_scores

## Compare Multiple Models

In [ ]:
def compare_model_outputs(judge, model_outputs, ground_truths):
    """
    Compare outputs from multiple models.
    
    Args:
        judge: ReportJudge instance
        model_outputs: Dict of {model_name: [reports]}
        ground_truths: List of ground truth data
    
    Returns:
        dict: {model_name: average_scores}
    """
    results = {}
    
    for model_name, reports in model_outputs.items():
        print(f"\nEvaluating {model_name}...")
        
        model_scores = []
        for report, gt in zip(reports, ground_truths):
            scores = judge.evaluate(report, gt)
            model_scores.append(scores)
        
        # Compute averages
        avg_scores = {
            'clinical_accuracy': sum(s['clinical_accuracy']['score'] for s in model_scores) / len(model_scores),
            'evidence_grounding': sum(s['evidence_grounding']['score'] for s in model_scores) / len(model_scores),
            'completeness': sum(s['completeness']['score'] for s in model_scores) / len(model_scores),
            'clarity': sum(s['clarity']['score'] for s in model_scores) / len(model_scores),
            'caution': sum(s['caution']['score'] for s in model_scores) / len(model_scores),
            'overall': sum(s['overall_score'] for s in model_scores) / len(model_scores)
        }
        
        results[model_name] = avg_scores
    
    return results

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_comparison(model_scores):
    """Plot comparison of model scores."""
    
    criteria = ['clinical_accuracy', 'evidence_grounding', 'completeness', 'clarity', 'caution']
    models = list(model_scores.keys())
    
    x = np.arange(len(criteria))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i, model in enumerate(models):
        scores = [model_scores[model][c] for c in criteria]
        ax.bar(x + i*width, scores, width, label=model)
    
    ax.set_xlabel('Criteria')
    ax.set_ylabel('Score (1-5)')
    ax.set_title('LLM-as-Judge Evaluation: Model Comparison')
    ax.set_xticks(x + width)
    ax.set_xticklabels([c.replace('_', ' ').title() for c in criteria], rotation=45)
    ax.legend()
    ax.set_ylim(0, 5)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Create Comparison Table

In [ ]:
def create_comparison_table(model_scores):
    """Create comparison table for paper."""
    
    df = pd.DataFrame(model_scores).T
    df = df.round(2)
    
    # Rename columns for paper
    df.columns = ['Clinical Accuracy', 'Evidence Grounding', 'Completeness', 'Clarity', 'Caution', 'Overall']
    
    print("\nMODEL COMPARISON TABLE:")
    print("="*80)
    print(df.to_string())
    print("="*80)
    
    # Save to CSV
    df.to_csv('results/model_comparison_table.csv')
    print("\n✅ Table saved to results/model_comparison_table.csv")
    
    return df